# 04 — Modeling and error analysis

**Cilj:** kompaktno uporediti modele A–I na istom splitu, prikazati feature importance, ablation i greške finalnog modela.

Modeli E i G imaju opcioni search blok. Podrazumevano se koriste razumni fiksni parametri da notebook ostane brz i ponovljiv; postavite `RUN_TUNING = True` za `RandomizedSearchCV`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

from src.features import (
    AVAILABILITY_FEATURES,
    CATEGORICAL_FEATURES,
    PER90_FEATURES,
    PERCENTAGE_FEATURES,
    RAW_NUMERIC_FEATURES,
)

RUN_TUNING = False
train_df = pd.read_csv(PROCESSED_DIR / "train_features.csv", index_col="split_index")
test_df = pd.read_csv(PROCESSED_DIR / "test_features.csv", index_col="split_index")
y_train = train_df["log_market_value"]
y_test = test_df["log_market_value"]

In [ ]:
raw_numeric = [*RAW_NUMERIC_FEATURES, *AVAILABILITY_FEATURES]
per90_numeric = [
    "Age", "Min", *[f"{f}_per90" for f in PER90_FEATURES],
    *PERCENTAGE_FEATURES, *AVAILABILITY_FEATURES,
]
shrunk_numeric = [
    "Age", "Min", *[f"{f}_per90_adjusted" for f in PER90_FEATURES],
    *PERCENTAGE_FEATURES, *AVAILABILITY_FEATURES,
]

def make_preprocessor(numeric, categorical, scale=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])

def evaluate(name, model, features):
    model.fit(train_df[features], y_train)
    prediction_log = model.predict(test_df[features])
    prediction_eur = np.expm1(prediction_log)
    metrics = {
        "Model": name,
        "MAE_log": mean_absolute_error(y_test, prediction_log),
        "RMSE_log": np.sqrt(mean_squared_error(y_test, prediction_log)),
        "R2": r2_score(y_test, prediction_log),
        "MAE_EUR": mean_absolute_error(np.expm1(y_test), prediction_eur),
    }
    return model, prediction_log, metrics

results = []
predictions = {}

## Models A–C — Ridge: raw, naive per-90, shrinkage

In [ ]:
ridge_specs = [
    ("A - Raw Ridge", raw_numeric),
    ("B - Naive Per90 Ridge", per90_numeric),
    ("C - Shrunk Per90 Ridge", shrunk_numeric),
]
ridge_models = {}
for name, numeric in ridge_specs:
    features = [*numeric, *CATEGORICAL_FEATURES]
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor(numeric, CATEGORICAL_FEATURES, scale=True)),
        ("model", Ridge(alpha=1.0)),
    ])
    fitted, pred, metrics = evaluate(name, pipeline, features)
    ridge_models[name] = fitted
    predictions[name] = pred
    results.append(metrics)

pd.DataFrame(results)

## Models D–E — Random Forest

In [ ]:
base_features = [*shrunk_numeric, *CATEGORICAL_FEATURES]
rf_base = Pipeline([
    ("preprocessor", make_preprocessor(shrunk_numeric, CATEGORICAL_FEATURES)),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])
rf_base, predictions["D - Random Forest"], metrics = evaluate(
    "D - Random Forest", rf_base, base_features
)
results.append(metrics)

rf_tuned = Pipeline([
    ("preprocessor", make_preprocessor(shrunk_numeric, CATEGORICAL_FEATURES)),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])
if RUN_TUNING:
    rf_tuned = RandomizedSearchCV(
        rf_tuned,
        {
            "model__n_estimators": [300, 500, 700],
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [0.6, 0.8, 1.0],
        },
        n_iter=12, scoring="neg_mean_absolute_error", cv=5,
        random_state=42, n_jobs=-1,
    )
else:
    rf_tuned.set_params(model__n_estimators=500, model__max_depth=20,
                        model__min_samples_leaf=2, model__max_features=0.8)

rf_tuned, predictions["E - Tuned Random Forest"], metrics = evaluate(
    "E - Tuned Random Forest", rf_tuned, base_features
)
results.append(metrics)

## Models F–G — XGBoost

In [ ]:
xgb_base = Pipeline([
    ("preprocessor", make_preprocessor(shrunk_numeric, CATEGORICAL_FEATURES)),
    ("model", XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42, n_jobs=-1,
    )),
])
xgb_base, predictions["F - XGBoost"], metrics = evaluate(
    "F - XGBoost", xgb_base, base_features
)
results.append(metrics)

xgb_tuned = Pipeline([
    ("preprocessor", make_preprocessor(shrunk_numeric, CATEGORICAL_FEATURES)),
    ("model", XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=-1)),
])
if RUN_TUNING:
    xgb_tuned = RandomizedSearchCV(
        xgb_tuned,
        {
            "model__n_estimators": [300, 500, 700, 1000],
            "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
            "model__max_depth": [3, 4, 5, 6],
            "model__min_child_weight": [1, 3, 5],
            "model__subsample": [0.7, 0.8, 0.9, 1.0],
            "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        },
        n_iter=20, scoring="neg_mean_absolute_error", cv=5,
        random_state=42, n_jobs=-1,
    )
else:
    xgb_tuned.set_params(
        model__n_estimators=700, model__learning_rate=0.03,
        model__max_depth=5, model__min_child_weight=3,
        model__subsample=0.8, model__colsample_bytree=0.8,
    )

xgb_tuned, predictions["G - Tuned XGBoost"], metrics = evaluate(
    "G - Tuned XGBoost", xgb_tuned, base_features
)
results.append(metrics)

## Feature importance — RF i XGBoost

In [ ]:
def feature_importance(fitted_pipeline, top_n=20):
    estimator = fitted_pipeline.best_estimator_ if hasattr(fitted_pipeline, "best_estimator_") else fitted_pipeline
    names = estimator.named_steps["preprocessor"].get_feature_names_out()
    values = estimator.named_steps["model"].feature_importances_
    return pd.DataFrame({"feature": names, "importance": values}).nlargest(top_n, "importance")

rf_importance = feature_importance(rf_tuned)
xgb_importance = feature_importance(xgb_tuned)
display(rf_importance)
display(xgb_importance)

## Models H–I — club context i prethodna vrednost

In [ ]:
xgb_params = dict(
    n_estimators=700, learning_rate=0.03, max_depth=5, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", random_state=42, n_jobs=-1,
)

categorical_h = ["Pos", "Comp", "MainSquad"]
features_h = [*shrunk_numeric, *categorical_h]
model_h = Pipeline([
    ("preprocessor", make_preprocessor(shrunk_numeric, categorical_h)),
    ("model", XGBRegressor(**xgb_params)),
])
model_h, predictions["H - XGBoost + Club"], metrics = evaluate(
    "H - XGBoost + Club", model_h, features_h
)
results.append(metrics)

numeric_i = [*shrunk_numeric, "log_previous_market_value", "previous_value_available"]
features_i = [*numeric_i, *categorical_h]
model_i = Pipeline([
    ("preprocessor", make_preprocessor(numeric_i, categorical_h)),
    ("model", XGBRegressor(**xgb_params)),
])
model_i, predictions["I - XGBoost + Previous Value"], metrics = evaluate(
    "I - XGBoost + Previous Value", model_i, features_i
)
results.append(metrics)

## Finalna tabela i grafikoni

In [ ]:
results_df = pd.DataFrame(results).sort_values("MAE_log").reset_index(drop=True)
display(results_df.style.format({
    "MAE_log": "{:.3f}", "RMSE_log": "{:.3f}", "R2": "{:.3f}", "MAE_EUR": "€{:,.0f}"
}))

plt.figure(figsize=(11, 4))
plt.bar(results_df["Model"], results_df["R2"])
plt.ylabel("R² on log target")
plt.xticks(rotation=45, ha="right")
plt.title("Model comparison")
plt.tight_layout()

## Ablation study

Poredi prethodnu tržišnu vrednost samu, performance + context (Model H), i njihov spoj (Model I).

In [ ]:
previous_numeric = ["log_previous_market_value", "previous_value_available"]
previous_only = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), previous_numeric)
    ])),
    ("model", XGBRegressor(**xgb_params)),
])
previous_only, previous_pred, previous_metrics = evaluate(
    "Previous Value Only", previous_only, previous_numeric
)

metric_by_model = {row["Model"]: row for row in results}
ablation_df = pd.DataFrame([
    previous_metrics,
    {**metric_by_model["H - XGBoost + Club"], "Model": "Performance + Context"},
    {**metric_by_model["I - XGBoost + Previous Value"], "Model": "Previous Value + Performance + Context"},
])
ablation_df

## Error analysis — Model I

In [ ]:
pred_i_log = predictions["I - XGBoost + Previous Value"]
error_analysis = test_df[[
    "Player", "Age", "Pos", "Comp", "MainSquad", "Min",
    "market_value_in_eur", "previous_market_value",
]].copy()
error_analysis["predicted_value_eur"] = np.expm1(pred_i_log)
error_analysis["error_eur"] = error_analysis["predicted_value_eur"] - error_analysis["market_value_in_eur"]
error_analysis["absolute_error_eur"] = error_analysis["error_eur"].abs()

display(error_analysis.nlargest(20, "absolute_error_eur"))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(error_analysis["market_value_in_eur"], error_analysis["predicted_value_eur"], alpha=0.6)
limit = max(error_analysis["market_value_in_eur"].max(), error_analysis["predicted_value_eur"].max())
axes[0].plot([0, limit], [0, limit], "--", color="black")
axes[0].set(xlabel="Actual €", ylabel="Predicted €", title="Actual vs predicted")
axes[1].scatter(error_analysis["Min"], error_analysis["absolute_error_eur"], alpha=0.6)
axes[1].set(xlabel="Minutes", ylabel="Absolute error €", title="Error vs playing time")
plt.tight_layout()